# Notebook 7: Thống kê cho báo cáo Task 2

Thu thập các số còn thiếu: kích thước mô hình, thời gian fine-tune, thống kê dữ liệu VietSpeech và FLEURS (số mẫu, giờ audio, phân bố độ dài, chia tập). **Chỉ đọc**, không sửa dữ liệu nào đã có; kết quả ghi vào `results/stats/` (bảng CSV, `summary.json`, hình PNG). Chạy được trên runtime CPU.

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os, re, json, glob
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

ROOT      = '/content/drive/MyDrive/project_DPL'
VS_META   = os.path.join(ROOT, 'datasets/vietspeech_100k/metadata_with_en_vinai.csv')
VS_HYP    = os.path.join(ROOT, 'results/asr_out/asr_hyp.csv')            # có cột dur_s (độ dài audio đọc từ file), split (dev/train)
VS_ZIPS   = os.path.join(ROOT, 'datasets/vietspeech_100k/audio_zips')
FL_META   = os.path.join(ROOT, 'datasets/fleurs_vi_en/fleurs_vi_voice_vi_en.csv')
FL_ZIPS   = os.path.join(ROOT, 'datasets/fleurs_vi_en/audio_zip')
FT_DIR    = os.path.join(ROOT, 'results/ft_out')
ASR_DIR   = os.path.join(ROOT, 'models/PhoWhisper-small')
MT_BASE   = os.path.join(ROOT, 'models/opus-mt-vi-en')
OUT       = os.path.join(ROOT, 'results/stats'); FIG = os.path.join(OUT, 'figs'); os.makedirs(FIG, exist_ok=True)
SUMMARY   = {}

def gb(b): return b / 2**30
def dir_size(d):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(d) for f in fs) if os.path.isdir(d) else None
def q(x): return {k: float(v) for k, v in zip(['min', 'p25', 'median', 'p75', 'p95', 'max', 'mean'], list(np.percentile(x, [0, 25, 50, 75, 95, 100])) + [np.mean(x)])}
def save(d, name): d.to_csv(os.path.join(OUT, name), index=False); return d
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## 1. Kích thước mô hình và thời gian fine-tune

In [20]:
rows = []
def add(name, d):
    s = dir_size(d)
    if s is None: print(f'Không thấy thư mục: {d}'); return
    files = [f for f in os.listdir(d) if f.endswith(('.safetensors', '.bin'))]
    w = sum(os.path.getsize(os.path.join(d, f)) for f in files)
    rows.append({'mô hình': name, 'thư mục (GB)': gb(s), 'file trọng số (MB)': w / 2**20})
add('OPUS-MT gốc (A)', MT_BASE)
for r in ['B_100', 'clean_100', 'B_50', 'B_25']: add(f'OPUS-MT {r}', os.path.join(FT_DIR, r, 'best_model'))
add('PhoWhisper-small', ASR_DIR)
sz = save(pd.DataFrame(rows), 'model_sizes.csv'); print(sz.round(2).to_string(index=False))
hub = glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--facebook--seamless-m4t-v2-large'))
if hub: print('Seamless (cache HF):', round(gb(dir_size(hub[0])), 2), 'GB'); SUMMARY['seamless_cache_gb'] = gb(dir_size(hub[0]))
else: print('Seamless: không có trong cache phiên này (kích thước xem trên trang mô hình).')

ft = []
for r in ['B_100', 'clean_100', 'B_50', 'B_25']:
    d = os.path.join(FT_DIR, r); tm = json.load(open(os.path.join(d, 'timing.json'))) if os.path.exists(os.path.join(d, 'timing.json')) else {}
    lh = json.load(open(os.path.join(d, 'log_history.json'))) if os.path.exists(os.path.join(d, 'log_history.json')) else []
    ev = [x for x in lh if 'eval_loss' in x]
    best = min(ev, key=lambda x: x['eval_loss']) if ev else {}
    ft.append({'run': r, 'epoch_chạy': len(ev), 'epoch_tốt_nhất': best.get('epoch'), 'dev_loss_tốt_nhất': best.get('eval_loss'),
               'thời_gian_phút': tm.get('train_s', np.nan) / 60, 'VRAM_đỉnh_GB': tm.get('peak_vram_gb')})
ft = save(pd.DataFrame(ft), 'finetune_summary.csv'); print(); print(ft.round(3).to_string(index=False))
SUMMARY['finetune_total_min'] = float(ft['thời_gian_phút'].sum())

          mô hình  thư mục (GB)  file trọng số (MB)
  OPUS-MT gốc (A)          0.27              273.57
    OPUS-MT B_100          0.27              273.57
OPUS-MT clean_100          0.27              273.57
     OPUS-MT B_50          0.27              273.57
     OPUS-MT B_25          0.27              273.57
 PhoWhisper-small          0.90              922.30
Seamless: không có trong cache phiên này (kích thước xem trên trang mô hình).

      run  epoch_chạy  epoch_tốt_nhất  dev_loss_tốt_nhất  thời_gian_phút  VRAM_đỉnh_GB
    B_100          11             6.0              1.464          23.766         7.838
clean_100           9             4.0              1.600          19.686         7.833
     B_50          10             5.0              1.547          12.478         7.835
     B_25           9             4.0              1.606           6.167         7.836


## 2. VietSpeech

In [21]:
meta = pd.read_csv(VS_META, keep_default_na=False)
hyp  = pd.read_csv(VS_HYP, keep_default_na=False)
ext = meta.audio_path.astype(str).str.extract(r'\.([A-Za-z0-9]+)$')[0].str.lower().value_counts().to_dict()
vs = {'metadata_rows': int(len(meta)), 'columns': meta.columns.tolist(), 'audio_ext': ext, 'sampled': int(len(hyp)),
      'dev': int((hyp.split == 'dev').sum()), 'train': int((hyp.split == 'train').sum())}
zs = glob.glob(os.path.join(VS_ZIPS, '*.zip')); vs['zip_total_gb'] = gb(sum(os.path.getsize(z) for z in zs)) if zs else None; vs['n_zips'] = len(zs)
d = hyp.dur_s.astype(float); d = d[d > 0]
vs['hours_sampled'] = float(d.sum() / 3600); vs['dur_s'] = q(d); vs['dur_gt30'] = int((d > 30).sum())
hyp['vi_words'] = hyp.vi_text.astype(str).str.split().str.len()
en = meta.set_index('id').en_text.astype(str)
hyp['en_words'] = hyp.id.map(en).astype(str).str.split().str.len()
vs['vi_words'] = q(hyp.vi_words); vs['en_words'] = q(hyp.en_words)
vs['asr_empty'] = int((hyp.asr_text.astype(str).str.strip() == '').sum()); vs['read_errors'] = int((hyp.error.astype(str).str.strip() != '').sum())
vs['nt_fixed'] = int(meta.en_text.astype(str).str.contains(r"\s+n['’]t\b").sum())
SUMMARY['vietspeech'] = vs
print(json.dumps(vs, ensure_ascii=False, indent=1))
split_tab = save(hyp.groupby('split').agg(mẫu=('id', 'size'), giờ=('dur_s', lambda x: x.sum() / 3600), từ_vi_TB=('vi_words', 'mean'), từ_en_TB=('en_words', 'mean')).reset_index(), 'vietspeech_splits.csv')
print(); print(split_tab.round(2).to_string(index=False))

{
 "metadata_rows": 100000,
 "columns": [
  "id",
  "source_index",
  "audio_path",
  "vi_text",
  "sampling_rate",
  "audio_format",
  "en_text"
 ],
 "audio_ext": {
  "audio": 100000
 },
 "sampled": 55000,
 "dev": 5500,
 "train": 49500,
 "zip_total_gb": null,
 "n_zips": 0,
 "hours_sampled": 61.19346166666667,
 "dur_s": {
  "min": 2.0,
  "p25": 2.72,
  "median": 3.616,
  "p75": 5.024,
  "p95": 7.188,
  "max": 8.0,
  "mean": 4.005390218181819
 },
 "dur_gt30": 0,
 "vi_words": {
  "min": 3.0,
  "p25": 10.0,
  "median": 13.0,
  "p75": 18.0,
  "p95": 27.0,
  "max": 44.0,
  "mean": 14.289781818181819
 },
 "en_words": {
  "min": 1.0,
  "p25": 7.0,
  "median": 10.0,
  "p75": 14.0,
  "p95": 21.049999999995634,
  "max": 211.0,
  "mean": 11.037581818181819
 },
 "asr_empty": 0,
 "read_errors": 0,
 "nt_fixed": 7430
}

split   mẫu   giờ  từ_vi_TB  từ_en_TB
  dev  5500  6.18     14.37     11.13
train 49500 55.01     14.28     11.03


## 3. FLEURS

In [22]:
fl = pd.read_csv(FL_META, keep_default_na=False)
fl['cluster'] = fl.split + '|' + fl.fleurs_id.astype(str)
fl['vi_words'] = fl.vi_script.astype(str).str.split().str.len(); fl['en_words'] = fl.en_script.astype(str).str.split().str.len()
dd = fl.duration_sec.astype(float)
fs = {'rows': int(len(fl)), 'columns': fl.columns.tolist(), 'sentences': int(fl.cluster.nunique()), 'hours': float(dd.sum() / 3600), 'dur_s': q(dd),
      'dur_gt30': int((dd > 30).sum()), 'match_status': fl.match_status.value_counts().to_dict(),
      'rec_per_sentence': q(fl.groupby('cluster').size()), 'vi_words': q(fl.vi_words), 'en_words': q(fl.en_words)}
zs = glob.glob(os.path.join(FL_ZIPS, '*.zip')); fs['zip_gb'] = gb(sum(os.path.getsize(z) for z in zs)) if zs else None
SUMMARY['fleurs'] = fs
print(json.dumps(fs, ensure_ascii=False, indent=1))
fl_tab = save(fl.groupby('split').agg(bản_ghi=('uid' if 'uid' in fl else 'fleurs_id', 'size'), câu=('cluster', 'nunique'), giờ=('duration_sec', lambda x: x.astype(float).sum() / 3600),
              dài_hơn_30s=('duration_sec', lambda x: int((x.astype(float) > 30).sum()))).reset_index(), 'fleurs_splits.csv')
print(); print(fl_tab.round(2).to_string(index=False))

{
 "rows": 4212,
 "columns": [
  "split",
  "fleurs_id",
  "audio_path",
  "vi_script",
  "en_script",
  "flores_id",
  "flores_split",
  "match_status",
  "sampling_rate",
  "duration_sec",
  "cluster",
  "vi_words",
  "en_words"
 ],
 "sentences": 1978,
 "hours": 13.2746,
 "dur_s": {
  "min": 2.52,
  "p25": 8.46,
  "median": 10.74,
  "p75": 13.32,
  "p95": 18.686999999999987,
  "max": 42.18,
  "mean": 11.345811965811965
 },
 "dur_gt30": 13,
 "match_status": {
  "matched": 4212
 },
 "rec_per_sentence": {
  "min": 1.0,
  "p25": 2.0,
  "median": 2.0,
  "p75": 3.0,
  "p95": 3.0,
  "max": 3.0,
  "mean": 2.129423660262892
 },
 "vi_words": {
  "min": 7.0,
  "p25": 22.0,
  "median": 28.0,
  "p75": 35.0,
  "p95": 48.0,
  "max": 75.0,
  "mean": 29.121320037986706
 },
 "en_words": {
  "min": 6.0,
  "p25": 16.0,
  "median": 20.0,
  "p75": 25.0,
  "p95": 33.0,
  "max": 60.0,
  "mean": 20.920465337132004
 },
 "zip_gb": 2.8491364140063524
}

     split  bản_ghi  câu  giờ  dài_hơn_30s
      test     

## 4. Hình

In [23]:
def hist(ax, x, title, xlabel, bins=40, vline=None):
    ax.hist(x, bins=bins, color='#4C72B0', edgecolor='white'); ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel('số mẫu')
    if vline: ax.axvline(vline, color='#C44E52', ls='--', lw=1); ax.text(vline, ax.get_ylim()[1] * .92, f' {vline}', color='#C44E52')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
hist(ax[0], d.clip(upper=45), f'VietSpeech: thời lượng ({len(d):,} mẫu)', 'giây (cắt ở 45)', vline=30)
hist(ax[1], dd.clip(upper=45), f'FLEURS: thời lượng ({len(dd):,} bản ghi)', 'giây (cắt ở 45)', vline=30)
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'duration_hist.png'), dpi=160); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
hist(ax[0], hyp.vi_words.clip(upper=80), 'VietSpeech: số từ tiếng Việt', 'từ (cắt ở 80)')
hist(ax[1], fl.vi_words.clip(upper=80), 'FLEURS: số từ tiếng Việt', 'từ (cắt ở 80)')
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'words_hist.png'), dpi=160); plt.close()

rps = fl.groupby('cluster').size()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(split_tab.split, split_tab['mẫu'], color='#4C72B0'); ax[0].set_title('VietSpeech: chia tập (mẫu)')
for i, v in enumerate(split_tab['mẫu']): ax[0].text(i, v, f'{v:,}', ha='center', va='bottom')
vc = rps.value_counts().sort_index(); ax[1].bar(vc.index.astype(str), vc.values, color='#55A868'); ax[1].set_title('FLEURS: số bản ghi trên mỗi câu'); ax[1].set_xlabel('số bản ghi'); ax[1].set_ylabel('số câu')
plt.tight_layout(); plt.savefig(os.path.join(FIG, 'splits_and_records.png'), dpi=160); plt.close()
print('Đã lưu 3 hình vào', FIG)

Đã lưu 3 hình vào /content/drive/MyDrive/project_DPL/results/stats/figs


In [24]:
json.dump(SUMMARY, open(os.path.join(OUT, 'summary.json'), 'w'), ensure_ascii=False, indent=1)
print('Đã ghi summary.json và các bảng CSV vào', OUT, ':', sorted(os.listdir(OUT)))

Đã ghi summary.json và các bảng CSV vào /content/drive/MyDrive/project_DPL/results/stats : ['figs', 'finetune_summary.csv', 'fleurs_splits.csv', 'model_sizes.csv', 'summary.json', 'vietspeech_splits.csv']


In [25]:
import pandas as pd
m = pd.read_csv('/content/drive/MyDrive/project_DPL/datasets/vietspeech_100k/metadata_with_en_vinai.csv', usecols=['sampling_rate', 'audio_format'])
print(m.sampling_rate.value_counts().head(), m.audio_format.value_counts().head(), sep='\n')

Series([], Name: count, dtype: int64)
audio_format
wav    100000
Name: count, dtype: int64
